# One-Port S11 Sweep

Measures S11 on a chosen set of MM4250 RF channels and saves each one as a `.s1p` file plus a QCoDeS run.

Uses `oneport_db_sweep.py` and `vna_measure.py`, which must sit in the **same folder as this notebook**. Outputs land in that folder too:

```
<this folder>/Sweeps/<date>_<temp>/<serials>/raw/RF<n>.s1p
<this folder>/mm4250_oneport.db
```

Raw acquisition only -- no calibration or de-embedding is applied.

**Instruments:** the cell below connects its own `ksvna`/`switch`. If you'd rather sweep from `QCodesMeasurmentFramework.ipynb`'s kernel, where those already exist, skip that cell and call `run_oneport_sweep(...)` there instead -- it finds instruments registered under those names on its own. Don't do both in one kernel: two live `MM4250("switch")` instances collide, and the switch's USB connection is exclusive.

In [ ]:
import sys
from pathlib import Path

# The drivers live in whichever repo root is above this notebook -- this
# folder if it's the mm4250-switch-sweep repo, or the framework repo if
# these files were copied into users/<name>/. Walk up until we find it.
try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))       # vna_measure / oneport_db_sweep live here

root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))       # drivers/ live here
print(f"notebook folder: {here}")
print(f"drivers from:    {root}")

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from vna_measure import setup_sweep, sweep_settings, measure_s11
from oneport_db_sweep import run_oneport_sweep

In [ ]:
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

In [ ]:
# Sweep setup -- only the settings you pass are changed.
setup_sweep(start=1e9, stop=10e9, points=1001, if_bandwidth=1e3, power=-20, vna=ksvna)

In [ ]:
# A single measurement, nothing saved:
freq, s11 = measure_s11(3, vna=ksvna, switch=switch)

In [ ]:
# Edit these for each run, then save the whole set:
channels = [1, 3, 5]         # any subset of 1-6; use list(range(1, 7)) for all of them
date_str = "YYYYMMDD"        # e.g. "20260917"
temp_str = "295K"            # e.g. "295K", "3K", "25mK"
switch_serials = "SN0001"    # the switch under test, e.g. "0030"

sweep_dir = run_oneport_sweep(channels, date_str, temp_str, switch_serials, vna=ksvna, switch=switch)
print(f"Sweep complete, .s1p files saved under {sweep_dir}")

In [ ]:
ksvna.close()
switch.close()